In [4]:
import re
import json
import statistics
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime as dt_datetime
from dotenv import load_dotenv
import os

## Carrega variáveis do .env

In [5]:
load_dotenv()

True

## Categorizar Log

In [6]:
# ---------------------------------------------------------------------------
# Linha base de todo evento estruturado
# ---------------------------------------------------------------------------
LINHAS_BASE = re.compile(
    r"^(?P<ts>\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}) "
    r"(?P<level>INFO|WARN|ERROR|DEBUG) "
    r"(?P<component>[\w$.]+): "
    r"(?P<msg>.*)$"
)

# ---------------------------------------------------------------------------
# Máscara de dados sensíveis - host/IP e paths s3a://
# ---------------------------------------------------------------------------
HOST_PATTERN = re.compile(r"\[?([0-9a-fA-F:\.]{7,})\]?(?::\d+)?")
PATH_PATTERN = re.compile(r"s3a?://[^\s,)\"]+")


class Sanitizador:
    def __init__(self):
        self.host_map: dict[str, str] = {}
        self.path_map: dict[str, str] = {}

    def mask_host(self, host: str) -> str:
        m = re.match(r"\[?([0-9a-fA-F:\.]+?)\]?(?::\d+)?$", host)
        normalized = m.group(1) if m else host
        if normalized not in self.host_map:
            self.host_map[normalized] = f"host_{len(self.host_map) + 1}"
        return self.host_map[normalized]

    def mask_paths(self, text: str) -> str:
        def _replace(m):
            raw = m.group(0)
            if raw not in self.path_map:
                self.path_map[raw] = f"path_{len(self.path_map) + 1}"
            return self.path_map[raw]
        return PATH_PATTERN.sub(_replace, text)

    def mask_text(self, text: str) -> str:
        text = self.mask_paths(text)

        def _replace_host(m):
            return f"[{self.mask_host(m.group(1))}]"
        return HOST_PATTERN.sub(_replace_host, text)


# ---------------------------------------------------------------------------
# Padrões de evento
# ---------------------------------------------------------------------------
CALL_SITE_PATTERN = re.compile(r"^(?P<metodo>\S+) at (?P<arquivo>[^:]+):(?P<linha>\d+)$")


def extrair_call_site(action: str) -> dict | None:
    """
    Decompõe o call site que o Spark registra em todo job (ex: "count at
    TransformacaoApolice.scala:145") em método/arquivo/linha do código do
    usuário. Retorna None quando o formato não bate (ex: ações sem call
    site legível, comuns em alguns planos gerados internamente).
    """
    m = CALL_SITE_PATTERN.match(action.strip())
    if not m:
        return None
    return {
        "metodo": m.group("metodo"),
        "arquivo": m.group("arquivo"),
        "linha": int(m.group("linha")),
    }


PADROES_DE_EVENTOS = {
    "job_start": re.compile(r"^Got job (?P<job_id>\d+) \((?P<action>.*?)\) with (?P<partitions>\d+) output partitions$"),
    "job_finish": re.compile(r"^Job (?P<job_id>\d+) finished: .*?, took (?P<duration_s>[\d.]+) s$"),
    "stage_submit": re.compile(r"^Submitting (?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+)"),
    "stage_finish": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) finished in (?P<duration_s>[\d.]+) s$"),
    "taskset_removed": re.compile(r"^Removed TaskSet (?P<stage_id>\d+)\.(?P<attempt>\d+), whose tasks have all completed"),
    "stage_failed": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) failed"),
    "task_start": re.compile(
        r"^Starting task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) "
        r"\((?P<host>[^,]+), executor (?P<executor>\d+)"
    ),
    "task_finish": re.compile(
        r"^Finished task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) "
        r"in (?P<duration_ms>\d+) ms on (?P<host>[^\s(]+) \(executor (?P<executor>\d+)\)"
    ),
    "broadcast_stored": re.compile(
        r"^Block broadcast_(?P<broadcast_id>\d+)(?:_piece\d+)? stored as (?:values|bytes) in memory "
        r"\(estimated size (?P<size_val>[\d.]+) (?P<size_unit>\w+),(?: actual size: [\d.]+ \w+,)? "
        r"free (?P<free_val>[\d.]+) (?P<free_unit>\w+)\)$"
    ),
    "block_added": re.compile(
        r"^Added broadcast_(?P<broadcast_id>\d+)_piece\d+ in memory on "
        r"(?P<host>\[[0-9a-fA-F:\.]+\]:\d+) "
        r"\(size: (?P<size_val>[\d.]+) (?P<size_unit>\w+), free: (?P<free_val>[\d.]+) (?P<free_unit>\w+)\)$"
    ),
    "shuffle_map_output_request": re.compile(r"^Asked to send map output locations for shuffle (?P<shuffle_id>\d+)$"),
    "shuffle_partition_advisory": re.compile(r"^For shuffle\((?P<shuffle_id>\d+)\), advisory target size: (?P<advisory_bytes>\d+)"),
    "executor_backlog_request": re.compile(r"^Requesting (?P<num_requested>\d+) new executors because tasks are backlogged"),
    "executor_registered": re.compile(r"^New executor (?P<executor>\d+) has registered \(new total is (?P<total>\d+)\)$"),
    "executor_not_found": re.compile(r"^No executor found for (?P<host>[0-9a-fA-F:\.]+)$"),
    "executor_lost": re.compile(r"^Lost executor (?P<executor>\d+) on (?P<host>[^:]+): (?P<reason>.+)$"),
    "app_final_status": re.compile(r"^SparkContext is stopping with exitCode (?P<exit_code>\d+)\.?$"),

}


@dataclass
class LogEvent:
    ts: str
    level: str
    component: str
    event_type: str
    data: dict = field(default_factory=dict)


# ---------------------------------------------------------------------------
# AJUSTE 5: `parse_log` agora recebe qualquer iterável de linhas (inclusive
# um objeto de arquivo aberto), em vez de exigir a string inteira já em
# memória. A lógica interna não muda uma linha sequer — só o tipo de entrada.
# ---------------------------------------------------------------------------
def parse_log(linhas, sanitizador: Sanitizador | None = None) -> list[LogEvent]:
    if sanitizador is None:
        sanitizador = Sanitizador()

    eventos: list[LogEvent] = []
    pendente_stack_trace: LogEvent | None = None

    for linha_bruta in linhas:
        linha = linha_bruta.rstrip("\n")
        m = LINHAS_BASE.match(linha.strip())

        if not m:

            if pendente_stack_trace is not None and linha.strip():
                pendente_stack_trace.data.setdefault("stack_trace", [])
                pendente_stack_trace.data["stack_trace"].append(
                    sanitizador.mask_text(linha.strip())
                )
            continue

        ts, level, component, msg = m.group("ts", "level", "component", "msg")
        pendente_stack_trace = None

        matched = False
        for event_type, pattern in PADROES_DE_EVENTOS.items():
            em = pattern.match(msg)
            if em:
                data = em.groupdict()
                if "host" in data and data["host"]:
                    data["host_masked"] = sanitizador.mask_host(data.pop("host"))
                if event_type == "job_start" and data.get("action"):
                    call_site = extrair_call_site(data["action"])
                    if call_site:
                        data["call_site"] = call_site
                evento = LogEvent(ts, level, component, event_type, data)
                eventos.append(evento)
                matched = True
                break

        if not matched and level in ("WARN", "ERROR"):
            evento = LogEvent(ts, level, component, "raw_warn_error",
                               {"msg": sanitizador.mask_text(msg)})
            eventos.append(evento)
            pendente_stack_trace = evento

    return eventos


def mapear_stage_para_origem(eventos: list[LogEvent]) -> dict[str, dict]:
    """
    Liga cada stage_id ao job (e, portanto, à linha do script) que o
    originou. Usa a ORDEM CRONOLÓGICA do log: o DAGScheduler do Spark
    processa um job por vez, então todo 'stage_submit' pertence ao job
    mais recente iniciado antes dele (o próximo "Got job N").

    PREMISSA (documentar na METODOLOGIA.MD): isso assume execução
    sequencial de jobs no driver. Em cenários de jobs concorrentes
    (múltiplas threads disparando actions em paralelo, scheduler FAIR
    com pools), essa ligação pode ficar imprecisa e deve ser tratada
    como "melhor esforço", não como verdade absoluta.
    """
    mapa: dict[str, dict] = {}
    job_atual: dict | None = None

    for e in eventos:
        if e.event_type == "job_start":
            job_atual = {
                "job_id": e.data.get("job_id"),
                "acao": e.data.get("action"),
                "call_site": e.data.get("call_site"),
            }
        elif e.event_type == "stage_submit" and job_atual is not None:
            mapa.setdefault(e.data["stage_id"], job_atual)

    return mapa


def agregado_por_stage(eventos: list[LogEvent]) -> list[dict]:
    tasks: dict[str, dict] = {}
    for e in eventos:
        if e.event_type in ("task_start", "task_finish"):
            tasks.setdefault(e.data["tid"], {}).update(e.data)

    by_stage: dict[str, list[dict]] = defaultdict(list)
    for t in tasks.values():
        if "duration_ms" in t and "stage_id" in t:
            stage_id = t["stage_id"].split(".")[0]
            by_stage[stage_id].append(t)

    # -----------------------------------------------------------------
    # AJUSTE 2: contagem de retries por estágio.
    # `task_id` vem no formato "índice.tentativa" (ex: "0.1" = task 0,
    # 2ª tentativa). Tentativa > 0 = a task foi refeita (retry).
    # -----------------------------------------------------------------
    retries_por_stage: dict[str, int] = defaultdict(int)
    for t in tasks.values():
        task_id = t.get("task_id", "")
        stage_id_t = t.get("stage_id", "").split(".")[0]
        partes = task_id.split(".")
        if len(partes) == 2 and partes[1].isdigit() and int(partes[1]) > 0:
            retries_por_stage[stage_id_t] += 1

    origem_por_stage = mapear_stage_para_origem(eventos)

    stage_meta = {}
    for e in eventos:
        if e.event_type == "stage_finish":
            stage_meta[e.data["stage_id"]] = {"duration_s": float(e.data["duration_s"]), "fonte": "stage_finish"}
        elif e.event_type == "taskset_removed" and e.data["stage_id"] not in stage_meta:
            stage_meta[e.data["stage_id"]] = {"duration_s": None, "fonte": "taskset_removed (sem duração precisa)"}

    summary = []
    for stage_id, task_list in by_stage.items():
        durations = [int(t["duration_ms"]) for t in task_list]
        per_executor = defaultdict(list)
        for t in task_list:
            per_executor[t.get("executor", "?")].append(int(t["duration_ms"]))

        durations_sorted = sorted(durations)
        p95_idx = max(0, int(len(durations_sorted) * 0.95) - 1)
        median = statistics.median(durations)

        summary.append({
            "stage_id": stage_id,
            "num_tasks": len(task_list),
            "duration_s": stage_meta.get(stage_id, {}).get("duration_s"),
            "duration_fonte": stage_meta.get(stage_id, {}).get("fonte"),
            "task_duration_ms": {
                "min": min(durations),
                "max": max(durations),
                "mean": round(statistics.mean(durations), 1),
                "median": median,
                "p95": durations_sorted[p95_idx],
            },
            "tasks_per_executor": {ex: len(v) for ex, v in per_executor.items()},
            "executor_duration_ms": {
                ex: {"mean": round(statistics.mean(v), 1), "total": sum(v)}
                for ex, v in per_executor.items()
            },
            "skew_ratio": round(max(durations) / median, 2) if median > 0 else None,
            "task_retries": retries_por_stage.get(stage_id, 0),
            "origem": origem_por_stage.get(stage_id),
        })

    return sorted(summary, key=lambda s: int(s["stage_id"]))


def resumo_eventos_esparsos(eventos: list[LogEvent]) -> dict:
    contagem = defaultdict(int)
    for e in eventos:
        contagem[e.event_type] += 1
    return dict(sorted(contagem.items(), key=lambda kv: -kv[1]))


# ---------------------------------------------------------------------------
# AJUSTE 1: eventos que já eram parseados (stage_failed, executor_lost) mas
# nunca chegavam ao JSON final. Agora viram métricas de fato.
# ---------------------------------------------------------------------------
def eventos_de_falha(eventos: list[LogEvent]) -> dict:
    stage_failures = [asdict(e) for e in eventos if e.event_type == "stage_failed"]
    executor_lost = [asdict(e) for e in eventos if e.event_type == "executor_lost"]
    return {
        "stage_failures": {"total": len(stage_failures), "detalhes": stage_failures},
        "executor_lost": {"total": len(executor_lost), "detalhes": executor_lost},
    }


# ---------------------------------------------------------------------------
# AJUSTE 3: duração total da execução, hoje ausente no nível "aplicação".
# Calculada pela diferença entre o primeiro e o último timestamp do log.
# ---------------------------------------------------------------------------
def duracao_total_execucao(eventos: list[LogEvent]) -> dict:
    timestamps = []
    for e in eventos:
        try:
            timestamps.append(dt_datetime.strptime(e.ts, "%d/%m/%y %H:%M:%S"))
        except ValueError:
            continue

    if not timestamps:
        return {"duration_s": None, "fonte": "nenhum timestamp pôde ser interpretado"}

    inicio, fim = min(timestamps), max(timestamps)
    return {
        "duration_s": (fim - inicio).total_seconds(),
        "inicio": inicio.strftime("%d/%m/%y %H:%M:%S"),
        "fim": fim.strftime("%d/%m/%y %H:%M:%S"),
        "fonte": "diferença entre o primeiro e o último timestamp do log",
    }


# ---------------------------------------------------------------------------
# AJUSTE 4: métricas pedidas pelo desafio que este log NÃO permite calcular,
# dado o nível de verbosidade (ver METODOLOGIA.MD). Ficam explícitas no JSON
# em vez de simplesmente não existirem, com a justificativa registrada.
# ---------------------------------------------------------------------------
def metricas_nao_disponiveis() -> dict:
    return {
        "gc_time_ratio": {
            "valor": None,
            "motivo": "Log em nível INFO padrão, sem eventos de GC verboso (flag -verbose:gc não habilitada).",
        },
        "shuffle_read_write_bytes": {
            "valor": None,
            "motivo": "Log expõe apenas sinal indireto de shuffle (advisory target size), não bytes lidos/escritos reais.",
        },
        "spill_memoria_disco": {
            "valor": None,
            "motivo": "Não há eventos de spill no log neste nível de verbosidade.",
        },
    }


if __name__ == "__main__":
    path = os.getenv("LOG_TRU_APOLICE_PREREFAT")

    if not path:
        raise ValueError("A variável não foi encontrada no arquivo .env.")

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Arquivo não encontrado!")

    sanitizador = Sanitizador()

    # AJUSTE 5 (continuação): o arquivo é passado como objeto aberto
    # diretamente para parse_log — nunca vira uma única string gigante
    # em memória via .read().
    with open(path, "r", encoding="utf-8") as f:
        parsed = parse_log(f, sanitizador)

    stage_summary = agregado_por_stage(parsed)
    raw_eventos = [asdict(e) for e in parsed if e.event_type == "raw_warn_error"]
    app_status = [asdict(e) for e in parsed if e.event_type == "app_final_status"]
    falhas = eventos_de_falha(parsed)
    duracao_total = duracao_total_execucao(parsed)

    output = {
        "duracao_total_execucao": duracao_total,
        "stages": stage_summary,
        "task_retries_total": sum(s["task_retries"] for s in stage_summary),
        "stage_failures": falhas["stage_failures"],
        "executor_lost": falhas["executor_lost"],
        "eventos_esparsos": resumo_eventos_esparsos(parsed),
        "metricas_nao_disponiveis": metricas_nao_disponiveis(),
        "status_final": app_status,
        "raw_warn_error": raw_eventos,
    }

    print(json.dumps(output, indent=2, ensure_ascii=False))

    with open("LOG_TRU_APOLICE_PREREFAT.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print("Arquivo 'log_agregado.json' gerado com sucesso.")
    print(f"Hosts mascarados: {len(sanitizador.host_map)} | Paths mascarados: {len(sanitizador.path_map)}")

{
  "duracao_total_execucao": {
    "duration_s": 362.0,
    "inicio": "26/08/18 06:09:21",
    "fim": "26/08/18 06:15:23",
    "fonte": "diferença entre o primeiro e o último timestamp do log"
  },
  "stages": [
    {
      "stage_id": "0",
      "num_tasks": 1,
      "duration_s": 4.367,
      "duration_fonte": "stage_finish",
      "task_duration_ms": {
        "min": 2370,
        "max": 2370,
        "mean": 2370,
        "median": 2370,
        "p95": 2370
      },
      "tasks_per_executor": {
        "3": 1
      },
      "executor_duration_ms": {
        "3": {
          "mean": 2370,
          "total": 2370
        }
      },
      "skew_ratio": 1.0,
      "task_retries": 0,
      "origem": null
    },
    {
      "stage_id": "2",
      "num_tasks": 50,
      "duration_s": 2.837,
      "duration_fonte": "stage_finish",
      "task_duration_ms": {
        "min": 61,
        "max": 2794,
        "mean": 1108.6,
        "median": 126.0,
        "p95": 2792
      },
      "tasks_